In [1]:
from bs4 import BeautifulSoup as bs
import pandas as pd
import numpy as np
import os
import io
import builtins
import time
import re
from itertools import product
import dash
from dash import dcc
from dash import html 
from dash import dash_table
import dash_bootstrap_components as dbc
from datetime import datetime


from dash.dependencies import Input, Output
pd.set_option('display.max_rows', 150)
pd.set_option('display.max_columns', 63)

In [2]:
python_exports = "C:/Users/jackj/Documents/Sports Interactive/Football Manager 2024/Python Exports"
current_save = "Man U"
all_players_file = "all_players_data.csv"

In [3]:
folder_path = python_exports + "/" + current_save
all_players_file_path = folder_path + "/" + all_players_file
all_folder_files = os.listdir(folder_path)


In [4]:
all_html_files = [f for f in all_folder_files if f.endswith('.html')]    
all_html_files = [w.replace('.html', '') + ' 00:00:01' for w in all_html_files]


In [5]:
if not os.path.exists(all_players_file_path):
    exported_dates = pd.DataFrame({'Date': [1, 2]})

else:
    # If the file exists, read the existing data and make a copy    
    exported_dates = pd.read_csv(all_players_file_path)

exported_dates = exported_dates.Date.unique().tolist()


In [6]:
for file in all_html_files:
    if file not in exported_dates:
        file_name = file.replace(' 00:00:01','')
        full_path = folder_path + '/' + file_name + '.html'
        fm_export= bs(open(full_path,encoding = "utf8"), "html.parser").find('table')
        raw_list = pd.read_html(io.StringIO(str(fm_export)))
        raw_table = pd.DataFrame(raw_list[0])
        date = file_name.split('.')[0] +  ' 00:00:01'
        raw_table['Date'] = pd.to_datetime(date, format = "%Y-%m-%d %H:%M:%S")
        raw_table['DoB'] = pd.to_datetime(raw_table["DoB"].str.split(" ").str[0], format="%d/%m/%Y")
        raw_table['DoB'] = raw_table['DoB'] + pd.Timedelta(seconds=1)
        raw_table['Age'] = (raw_table['Date']-raw_table['DoB']).dt.days/365.25
        raw_table['Acc'] = pd.to_numeric(raw_table['Acc'], errors='coerce')

        raw_table = raw_table.dropna(subset=['Acc'])
        if not os.path.exists(all_players_file_path):
            raw_table.to_csv(all_players_file_path, index=False)
            print(f"Created file: {all_players_file_path}")
        else:
            # If the file exists, read the existing data and make a copy
            timestr = time.strftime("%Y%m%d-%H%M%S")
            existing_df = pd.read_csv(all_players_file_path)
            csv_filename = f'C:/Users/jackj/Documents/Sports Interactive/Football Manager 2024/Python Exports/{current_save}/all_players_data_{timestr}.csv'
            existing_df.to_csv(csv_filename, index=False)

            # Union the existing DataFrame with the new DataFrame
            combined_df = pd.concat([existing_df, raw_table]).drop_duplicates().reset_index(drop=True)

            # Save the combined DataFrame back to the CSV file
            combined_df.to_csv(all_players_file_path, index=False)
            print(f"Updated file: {all_players_file_path}")


Created file: C:/Users/jackj/Documents/Sports Interactive/Football Manager 2024/Python Exports/Man U/all_players_data.csv


In [7]:
full_path = folder_path + '/' + 'all_players_data' + '.csv'

In [8]:
raw_table = pd.read_csv(full_path)


In [9]:
def process_segment(segment):
    segment = segment.replace("D/","Z/").replace("D ",("Z "))
    sides = re.findall(r'\((.*?)\)', segment)
    sides = sides[0]    
    lines = segment.replace("("+sides+")","").replace("/","").replace("AM","A").replace("DM","O").replace('ST','F').replace("WB","O").replace("Z","D").strip()
    result = [f"{x}{y}" for x, y in product(lines, sides)] 
    return result

In [10]:
def combine_segment(segments):
    result = []
    segments = segments.replace('GK','G (C)')
    segments = re.sub(r'\bDM\b(,?)', r'DM (C)\1', segments)
    for segment in segments.split(','):
        result.extend(process_segment(segment.strip()))
    return result

In [11]:
def apply_combinations(df, column):

    df["Positions"] = df[column].apply(combine_segment)
    df['Positions'] = df['Positions'].apply(replace_values)

    return df


In [12]:
def replace_values(lst):
    new_lst = []
    for item in lst:
        item = item.replace('O', 'DM')
        item = item.replace('GC', 'GK')
        item = item.replace('FC', 'ST')
        item = item.replace('A', 'AM')
        item = item.replace('DMR', 'WBR')
        item = item.replace('DML', 'WBL')
        item = item.replace('DMC', 'DM')


        new_lst.append(item)
    return new_lst

In [13]:
df = apply_combinations(raw_table, 'Position')
a=1

In [14]:
unique_values = set([item for sublist in df['Positions'] for item in sublist])


In [15]:
for position in unique_values:
    df[position] = df.apply(lambda x: 1 if position in x["Positions"] else 0, axis=1)


In [16]:
latest_export_date = df['Date'].max()
df['gk_rating'] = ((5 * (df['Agi']+df['Ref']+ df['Cmd']+df['Cnt'])) + (3 * (df['1v1']+df['Ant']+df['Kic']+df['Pos'])) +(1 * (df['Acc']+df['Aer']+df['Dec']+df['Fir']+df['Han']+df['Pas']+df['TRO']+df['Vis'])))/40
df['of_rating'] = (((df['Acc']+ df['Pac'])*5) +  ((df['Dri']+ df['Ant'] + df['Cnt']+ df['Agi']+df['Bal']+ df['Jum'])*3) + (df['Det']+ df['Wor'] + df['Sta']+ df['Nat.1']))/32
df['rating'] = df['gk_rating'].where(df['Positions'] == 'GK', df['of_rating'])
df['physical'] = df['Acc']+df['Pac'] + df['Jum']
earliest_ratings = df.loc[df.groupby('UID')['Date'].idxmin(), ['UID', 'rating','physical','CA','PA']].rename(columns={'rating': 'first_rating','physical': 'first_physical','CA': 'first_CA','PA': 'first_PA'})
df = df.merge(earliest_ratings, on='UID', how='left')

df['rating_growth'] = df['rating'] - df['first_rating']
df['physical_growth'] = df['physical'] - df['first_physical']
df['CA_growth'] = df['CA'] - df['first_CA']

df['age_at_export'] = df['Age'].apply(np.floor)
df['current_actual_age'] = df.groupby(['UID'])['age_at_export'].transform('last')
df['is_latest_export'] = df['Date'] == latest_export_date
df_max_date = df[df['Date'] == latest_export_date]
df_max_date['in_latest_export'] = True
df = pd.merge(df, df_max_date, how="left", on=["UID"],suffixes = [None,'_current'])
df=df.round(2)




KeyError: "['CA', 'PA'] not in index"

In [ ]:
import pandas as pd
from datetime import datetime

# Sample dataframe (ensure 'DoB' and 'date' are in datetime format)
# df = pd.DataFrame({'DoB': [...], 'date': [...]})

# Convert 'DoB' and 'date' to datetime if they are not in datetime format
df['DoB'] = pd.to_datetime(df['DoB'])
df['Date'] = pd.to_datetime(df['Date'])

# Function to calculate the most recent 1st of July based on the reference 'date'
def get_start_of_season(date):
    year = date.year
    
    # Check if the reference 'date' is before or after July 1st of that year
    if date.month < 7 or (date.month == 7 and date.day < 1):
        # If before 1st July, use the previous year's 1st of July
        return pd.Timestamp(f'{year-1}-07-01')
    else:
        # If after or on 1st July, use the current year's 1st of July
        return pd.Timestamp(f'{year}-07-01')

# Apply the function to calculate the most recent 1st of July for each row based on 'date'
df['start_of_season'] = df['Date'].apply(get_start_of_season)

# Calculate the age on the most recent 1st of July
df['start_of_season_age'] = df['start_of_season'].dt.year - df['DoB'].dt.year

# If their birthday hasn't occurred yet on that 1st July, subtract 1 from their age
df['start_of_season_age'] -= ((df['start_of_season'].dt.month < df['DoB'].dt.month) |
                               ((df['start_of_season'].dt.month == df['DoB'].dt.month) &
                                (df['start_of_season'].dt.day < df['DoB'].dt.day)))





In [ ]:
columns_to_check = ['GK', 'DC', 'WBR', 'WBL', 'DM', 'AMC', 'ST']
df['Positions'] = df.apply(lambda row: [col for col in columns_to_check if row[col] == 1], axis=1)
df['Positions'] = df['Positions'].apply(lambda x: ', '.join(x))
df['Positions_filter'] = df['Positions']+', All'
df['All'] = 1
team_df = df[df['Club'].str.contains(current_save, case=False, na=False)]
team_df = team_df[team_df['is_latest_export']==True]
df = df[['Name','Club','current_actual_age','Positions','Positions_filter','GK', 'DC', 'WBR', 'WBL', 'DM', 'AMC', 'ST','All','rating','rating_growth','physical','physical_growth','Pac','Acc','Jum','age_at_export','is_latest_export','in_latest_export','start_of_season_age']]

In [ ]:
import dash
import dash_table
import dash_core_components as dcc
import dash_html_components as html
import dash_bootstrap_components as dbc
from dash.dependencies import Input, Output
import pandas as pd

# Initialize the app
app = dash.Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP], suppress_callback_exceptions=True)

# Sample data (you can replace this with your actual data)
# df = pd.read_csv("your_data.csv")

# Define layout with tabs
app.layout = dbc.Container([
    dbc.Row(html.H1("Web App", style={'text-align': 'center', 'margin-bottom': '20px'})),
    
    dbc.Tabs([
        dbc.Tab(label="Data Table", tab_id="tab-table"),
        dbc.Tab(label="Charts", tab_id="tab-charts"),
    ], id="tabs", active_tab="tab-table"),
    
    html.Br(),
    
    dbc.Row([
        dbc.Col(html.Div([
            html.Label("Select Age Range"),
            dcc.RangeSlider(15, 40,
                            value=[15, 40],
                            id='age_range_slider',
                            marks={i: str(i) for i in range(15, 41, 5)}),
        ]), width=3),

        dbc.Col(html.Div([
            html.Label("Select Positions"),
            dcc.Dropdown(id="position_filter",
                         options=[{'label': pos, 'value': pos} for pos in ['GK', 'DC', 'WBR', 'WBL', 'DM', 'AMC', 'ST', 'All']],
                         multi=True,
                         value=['All']),
        ]), width=3),

        dbc.Col(html.Div([
            html.Label("Filter by Latest Export"),
            dcc.Dropdown(id="is_latest_export_filter",
                         options=[{'label': 'True', 'value': True}, {'label': 'False', 'value': False}],
                         multi=True,
                         value=[True]),
        ]), width=3),

        dbc.Col(html.Div([
            html.Label("Filter by Club Name"),
            dcc.Input(id="club_filter",
                      type='text',
                      placeholder='Enter a club name...',
                      debounce=True,
                      value=""),
        ]), width=3),
    ], className='row'),
    
    html.Br(),
    html.Div(id='tab-content')
])

# Callback to filter the data
def filter_data(selected_age_range, selected_positions, is_latest_export, club_filter):
    # Default selected_positions to empty list if None
    if selected_positions is None:
        selected_positions = []

    filtered_df = df.copy()
    filtered_df['Positions_filter'] = filtered_df['Positions_filter'].str.split(', ')
    
    if 'All' not in selected_positions:
        filtered_df = filtered_df[filtered_df['Positions_filter'].apply(lambda x: any(pos in x for pos in selected_positions))]
    
    filtered_df['Positions_filter'] = filtered_df['Positions_filter'].apply(lambda x: ', '.join(x))
    
    if is_latest_export is not None:  # Ensure is_latest_export is handled as well
        filtered_df = filtered_df[filtered_df['is_latest_export'].isin(is_latest_export)]
    
    filtered_df = filtered_df[(selected_age_range[0] <= filtered_df['start_of_season_age']) & (filtered_df['start_of_season_age'] <= selected_age_range[1])]
    
    if club_filter:
        filtered_df = filtered_df[filtered_df['Club'].str.contains(club_filter, case=False, na=False)]
    
    return filtered_df

# Callback to update content based on selected tab
@app.callback(Output('tab-content', 'children'),
              [Input('tabs', 'active_tab'),
               Input('age_range_slider', 'value'),
               Input('position_filter', 'value'),
               Input('is_latest_export_filter', 'value'),
               Input('club_filter', 'value')])
def update_tab_content(active_tab, selected_age_range, selected_positions, is_latest_export, club_filter):
    filtered_df = filter_data(selected_age_range, selected_positions, is_latest_export, club_filter)
    
    if active_tab == 'tab-charts':
        # Calculate the average rating growth and physical growth for each age group
        avg_rating_growth = filtered_df.groupby('start_of_season_age')['rating_growth'].mean().reset_index()
        avg_physical_growth = filtered_df.groupby('start_of_season_age')['physical_growth'].mean().reset_index()

        # Rating Growth chart with average rating growth per age group
        rating_growth_fig = {
            'data': [{'x': avg_rating_growth['start_of_season_age'], 'y': avg_rating_growth['rating_growth'], 'type': 'bar', 'name': 'Average Rating Growth'}],
            'layout': {'title': 'Average Rating Growth by Age', 'xaxis': {'title': 'Age'}}
        }
        
        # Physical Growth chart with average physical growth per age group
        physical_growth_fig = {
            'data': [{'x': avg_physical_growth['start_of_season_age'], 'y': avg_physical_growth['physical_growth'], 'type': 'bar', 'name': 'Average Physical Growth'}],
            'layout': {'title': 'Average Physical Growth by Age', 'xaxis': {'title': 'Age'}}
        }
        
        return dbc.Container([
            dbc.Row([
                dbc.Col(dcc.Graph(id='rating_growth_chart', figure=rating_growth_fig), width=6),
                dbc.Col(dcc.Graph(id='physical_growth_chart', figure=physical_growth_fig), width=6)
            ])
        ])
    
    # Default Data Table view for the "Data Table" tab
    return dbc.Container(
        dbc.Row(
            dbc.Col(
                html.Div(
                    id='output_container',
                    children=[
                        dash_table.DataTable(
                            columns=[{"name": i, "id": i} for i in ['Name', 'Club', 'start_of_season_age', 'Positions_filter', 'rating', 'rating_growth', 'physical', 'physical_growth']],
                            data=filtered_df.to_dict('records'),
                            page_size=15,
                            id='my_squad',
                            filter_action="native",
                            sort_action="native",
                            sort_mode="multi",
                            row_selectable="multi"
                        )
                    ]
                ),
            ),
            className='row'
        ), fluid=True
    )



In [ ]:
if __name__ == '__main__':
    app.run(debug=True,jupyter_mode="external")

In [ ]:
df